In [ ]:
import pandas as pd
df = pd.read_csv('c:/data/time/data2.csv')
df.head()
#################################
#전처리
# 문자열 변수 date 를 datetime 타입으로 변경
df['date'] = pd.to_datetime(df['date'])
# 이 날짜를 index로 설정
df = df.set_index('date')
df.head()
#################################
df.plot()
#################################
#인덱스가 0,1,2,3 ... 에서 날짜로 변경됨
#학습할 변수는 date
import matplotlib.pyplot as plt
split_date = pd.Timestamp('01-01-2011')
# 학습용: 2011/1/1까지의 자료
# 검증용: 이후 자료
train = df.loc[:split_date, ['price']]
test = df.loc[split_date:, ['price']]
ax = train.plot()
test.plot(ax=ax)
plt.legend(['train', 'test'])
#################################
from sklearn.preprocessing import MinMaxScaler
sc = MinMaxScaler()
sc.fit(train)
train_sc = sc.transform(train)
test_sc = sc.transform(test)
train_sc[:10]
#################################
#넘파이배열을 데이터프레임으로 변환
train_sc_df = pd.DataFrame(train_sc, columns=['Scaled'], index=train.index)
test_sc_df = pd.DataFrame(test_sc, columns=['Scaled'], index=test.index)
train_sc_df.head()
#################################
s = pd.Series([100, 200, 300])
s2 = s.shift(1)
print(s)
print(s2)
#################################
for s in range(1, 13):
    train_sc_df['shift_{}'.format(s)] = train_sc_df['Scaled'].shift(s)
    test_sc_df['shift_{}'.format(s)] = test_sc_df['Scaled'].shift(s)

train_sc_df.head(13)
#과거값 12개로 현재값을 예측하고자 함
#################################
#결측값 NaN 제거
#독립변수 : shift_1 ~ shift_12
X_train = train_sc_df.dropna().drop('Scaled', axis=1)
#종속변수 : Scaled
y_train = train_sc_df.dropna()[['Scaled']]
X_test = test_sc_df.dropna().drop('Scaled', axis=1)
y_test = test_sc_df.dropna()[['Scaled']]
#################################
#넘파이배열로 저장
X_train = X_train.values
X_test= X_test.values
y_train = y_train.values
y_test = y_test.values
#print(X_train.shape)
print(X_train)
#print(y_train_shape)
print(y_train)
#################################
# 케라스에 필요한 3차원 형태로 변환
# RNN에는 시간 개념이 있기 때문에 차원이 추가됨
# [size,timestep,변수개수]
X_train_t = X_train.reshape(X_train.shape[0], 12, 1)
X_test_t = X_test.reshape(X_test.shape[0], 12, 1)
print("최종 DATA")
print(X_train_t.shape)
print(X_train_t)
print(y_train)
#################################
from tensorflow.keras.layers import LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

model = Sequential()
model.add(Input(shape=(12, 1)))  # timestep=12, 변수=1
model.add(LSTM(20))              # units=20
model.add(Dense(1))
model.compile(loss='mse', optimizer='adam')
#################################
model.fit(X_train_t, y_train, epochs=250, batch_size=64, verbose=1)
#################################
score=model.evaluate(X_test_t, y_test, verbose=0)
print(score) #평균제곱오차
y_pred = model.predict(X_test_t)
print(y_pred.flatten()[:10])
print(y_test.flatten()[:10])
#################################
import numpy as np
#실제값-예측값의 평균값
np.mean(y_test - y_pred)
#################################
y_predicted = sc.inverse_transform(y_pred)
y_tested = sc.inverse_transform(y_test)
np.mean(y_tested - y_predicted)
#################################
import matplotlib.pyplot as plt
pred=model.predict(X_test_t)
a=np.concatenate((y_train.flatten(), np.zeros(len(y_test))+np.nan))
b=np.concatenate((np.zeros(len(y_train))+np.nan, pred.flatten()))
plt.plot(a, 'r-', label='real')
plt.plot(b, 'b-', label='pred')
plt.legend()
plt.show()




